# Proyecto sobre Redes Neuronales Artificiales (Bonificación opcional)

### Redes Neuronales Artificiales
### Feed-Forward Neural Networks - Perceptrón Multicapa (MLP)

### Universidad de Medellin

### Facultad de Ingenierías

### Ingeniería de Sistemas

### UdeM

#### Profesor: Antonio Tamayo Herrera

## Guía de la práctica

En esta archivo va a encontrar tanto celdas de código cómo celdas de texto con las instrucciones para desarrollar el laboratorio.

Lea atentamente las instrucciones entregadas en las celdas de texto correspondientes y proceda con la solución de las preguntas planteadas.

Nota: no olvide ir ejecutando las celdas de código de arriba hacia abajo para que no tenga errores de importación de librerías o por falta de definición de variables.

## Indicaciones

Este ejercicio tiene como objetivo implementar una red neuronal artificial bajo la arquitectura feed-forward neural network o perceptrón multicapa (MLP) para resolver un problema de clasificación multiclase.

Para la implementación del modelo feed-forward neural network haremos uso de la librería keras y tensorflow. (Para instalarla use: !pip install keras y !pip install tensorflow, encontrará las celdas con este código más adelante, descomente y ejecute, la instalación puede tardar varios minutos). Si lo prefiere, puede desarrollar el laboratorio en google colaboratory y no necesitará instalar ninguna libreríía.

Para el problema de clasificación usaremos la siguiente base de datos: https://archive.ics.uci.edu/ml/datasets/Cardiotocography

#### Abstract:
The dataset consists of measurements of fetal heart rate (FHR) and uterine contraction (UC) features on cardiotocograms classified by expert obstetricians.


#### Data Set Information:

2126 fetal cardiotocograms (CTGs) were automatically processed and the respective diagnostic features measured. The CTGs were also classified by three expert obstetricians and a consensus classification label assigned to each of them. Classification was both with respect to a morphologic pattern (A, B, C. ...) and to a fetal state (N, S, P). Therefore the dataset can be used either for 10-class or 3-class experiments.

#### Attribute Information:

LB - FHR baseline (beats per minute)

AC - # of accelerations per second

FM - # of fetal movements per second

UC - # of uterine contractions per second

DL - # of light decelerations per second

DS - # of severe decelerations per second

DP - # of prolongued decelerations per second

ASTV - percentage of time with abnormal short term variability

MSTV - mean value of short term variability

ALTV - percentage of time with abnormal long term variability

MLTV - mean value of long term variability

Width - width of FHR histogram

Min - minimum of FHR histogram

Max - Maximum of FHR histogram

Nmax - # of histogram peaks

Nzeros - # of histogram zeros

Mode - histogram mode

Mean - histogram mean

Median - histogram median

Variance - histogram variance

Tendency - histogram tendency

CLASS - FHR pattern class code (1 to 10)

NSP - fetal state class code (N=normal (1); S=suspect (2); P=pathologic (3))

En la siguiente celda de código se realiza la transformación de la variable de salida Y al formato One Hot Encoding.

In [ ]:
#!pip install keras

In [ ]:
#!pip install tensorflow

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
from __future__ import division

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder
import time

def oneHotEncoder(y):

    onehot_encoder = OneHotEncoder(sparse_output=False)
    integer_encoded = y.reshape(len(y), 1)
    onehot_encoded = onehot_encoder.fit_transform(integer_encoded)
    return onehot_encoded

#cargamos la bd de entrenamiento
db = np.loadtxt('/content/DB_Fetal_Cardiotocograms.txt',delimiter='\t')  # Assuming tab-delimiter

X = db[:,0:21]

Y = db[:,22]
Y = oneHotEncoder(Y)
print(Y)

La variable class tiene fuga de informacion porque es una variable de prediccion

In [ ]:
pd.DataFrame(X).head()

In [ ]:
pd.DataFrame(X).describe().T

In [ ]:
pd.isnull(X).sum()

## Ejercicio 1.

1.1. Por qué es necesario hacer la codificación One-Hot Encodign de la variable de salida?

R/: Sabemos que en la capa de salida se deben poner tantas neuronas como clases haya. Es necesario hacer One Hot Encoding para poder supervisar el modelo. Si lo predicho es de 3 componentes, lo teórico también debe tener 3 componentes. Debo transformar el vector para que coincidan el número de componentes.
    
1.2 De qué manera se toma la decisión para asignar la clase una muestra en la red feed-forward cuando hay múltiples clases?

R/:Se alica la función Softmax y a la clase de mayor probabilidad se le asigna esa muestra.

La siguiente celda tiene un método para calcular las medidas de error, según sea un problema biclase o multiclase, solo debe ejecutarla.

In [ ]:
def metrics(y_test, y_pred):
  '''
  Parametros:
    - lista (array) y_test (One-hot encoding)
    - lista (array) y_predichas (softmax vector)
  Return:
    - F1
    - Recall
    - Precision
    - Accuracy
    - Especificity
  '''
  y_test = np.asarray(y_test)
  y_pred = np.asarray(y_pred)

  if y_test[0].shape[0] == 1:    #Biclase
    TP,TN,FP,FN = 0,0,0,0
    for t,p in zip(y_test,y_pred):
      if t==1:
        if p>=0.5: #yp==1:
          TP+=1
        else:
          FN+=1
      else:
        if p>=0.5: #yp==1:
          FP+=1
        else:
          TN+=1

    recall = TP/(TP+FN)
    prec = TP/(TP+FP)
    acc = (TP+TN)/(TP+FP+FN+TN)
    f1 = (2*recall*prec)/(recall+prec)
    esp = TN/(TN+FP)

    return f1, recall, prec, acc, esp

  else:    #Multiclase
    error = 0
    for test,pred in zip(y_test,y_pred):
        t = np.argmax(test)
        p = np.argmax(pred)
        if t!=p:
          error+=1

    err = error/np.shape(y_test)[0]
    acc = 1 - err

    return acc


## Ejercicio 2

En la siguiente celda, complete el código donde le sea indicado. Consulte la documentación oficial de la librería keras para la arquitectura propuesta aquí: https://keras.io/getting-started/sequential-model-guide/

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation

#Para calcular el costo computacional
tiempo_i = time.time()

#Complete el código para crear la red neuronal artificial.
#Para la configuración de la arquitectura de la red tome como base lo que
#se pide en la tabla que encuentra despuéés de esta celda de código.
#Recuerde usar la función de activación SoftMax en la capa de salida. Para la capa oculta puede usar
#ReLu, tanh o sigmoid; se recomienda ReLu)
model = Sequential()
model.add(keras.Input(shape=(21,)))
model.add(Dense(10, activation="relu"))
model.add(Dense(3, activation="softmax"))

#Implemetamos la metodología de validación cross validation con 10 folds
Accuracy = np.zeros(10)

kf = KFold(n_splits=10, shuffle=True)
j = 0
for train_index, test_index in kf.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = Y[train_index], Y[test_index]

    #Complete el código para compilar el modelo
    #Use el optimizador Adam
    #Se utilizo categorical cross entropy ya que es un problema de clasificacion multiclase
    model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[
        keras.metrics.BinaryAccuracy(),
        keras.metrics.FalseNegatives(),
    ],
)


    #Complete el cóódigo para entrenar el modelo
    model.fit(
    x=X_train,
    y=y_train,
    batch_size=None,
    epochs=10,
    verbose="auto",
    callbacks=None,
    validation_split=0.0,
    validation_data=None,
    shuffle=True,
    class_weight=None,
    sample_weight=None,
    initial_epoch=0,
    steps_per_epoch=None,
    validation_steps=None,
    validation_batch_size=None,
    validation_freq=1,
)


    #Complete el código aquí
    #Validación del modelo
    y_pred = model.predict(X_test)

    Accuracy[j] = metrics(y_test, y_pred)
    j+=1

print("\nEficiencia en la clasificacióón: " + str(np.mean(Accuracy)) + " +/- " + str(np.std(Accuracy)))

print ("\n\nTiempo total de ejecución: " + str(time.time()-tiempo_i)+ " segundos.")


2.1 Varíe el número de épocas del algoritmo de entrenamiento y el número de neuronas de la capa oculta y complete la siguiente tabla.

<table>
  <tr>
    <th># de Epocas</th>
    <th># Neuronas en 1ra capa oculta</th>
    <th># Neuronas en 2da capa oculta</th>
    <th>Eficiencia +/- IC</th>
    <th>Tiempo de ejecución (S)</th>
  </tr>
  <tr>
    <td>10</td>
    <td>10</td>
    <td>N/A</td>
    <td>0.8471543095048277 +/- 0.02979405837721828</td>
    <td>46.08951663970947</td>
  </tr>
  <tr>
    <td>30</td>
    <td>10</td>
    <td>N/A</td>
    <td>0.8692621135618743 +/- 0.02268551151865591</td>
    <td>84.22432255744934</td>
  </tr>
    <tr>
    <td>50</td>
    <td>10</td>
    <td>N/A</td>
    <td>0.8843188059172645 +/- 0.01958212719825958</td>
    <td>132.4964861869812</td>
  </tr>
  <tr>
    <td>10</td>
    <td>10</td>
    <td>2</td>
    <td>0.7784657631322526 +/- 0.024401574692260803</td>
    <td>46.97368240356445</td>
  </tr>
  <tr>
    <td>30</td>
    <td>10</td>
    <td>2</td>
    <td>0.7784657631322526 +/- 0.033406451418436636</td>
    <td>89.71132826805115</td>
  </tr>
  <tr>
    <td>50</td>
    <td>10</td>
    <td>2</td>
    <td>0.8725396403578705 +/- 0.01830091361435189</td>
    <td>136.21040320396423</td>
  </tr>
</table>


2.2 Analice e interprete los resultados de la tabla anterior. ¿Cómo influye el cambio de los parámetros propuesto en la eficiencia y en el tiempo de ejecución? Explique.

R/: Número de épocas: entre más épocas se entrena la red, mejor clasifica, pero más se demora. Con una capa oculta, la eficiencia sube de 84.7 % (10 épocas) a 88.4 % (50 épocas), aunque el tiempo casi se triplica (de 46 a 132 segundos). Es decir, se gana precisión, pero cada mejora cuesta más tiempo que la anterior.

Segunda capa oculta: agregar una segunda capa de solo 2 neuronas empeoró el modelo. Con 10 y 30 épocas, la red sacó 77.8 %, que es justo el porcentaje de casos “Normales” en los datos. Solo con 50 épocas logró aprender algo (87.3 %), pero aun así quedó por debajo de la red de una sola capa, demorándose lo mismo.

Conclusión: la red con una sola capa oculta fue mejor en todos los casos. Si se quiere rapidez, basta con 10 épocas; si se quiere la mejor clasificación, conviene usar 50 épocas. Como se trata de diagnosticar el estado de un feto, vale la pena esperar un poco más para equivocarse menos.

## Ejercicio 3

Ahora va a resolver el problema de regresión de la base de datos AirQuality (https://archive.ics.uci.edu/ml/datasets/Air+Quality) usando RNA bajo la arquitectura feed-forward. En la siguiente celda no tiene que implementar código, solo ejecutarla y analizar que proceso de limpieza que se lleva a cabo.

In [ ]:
import numpy as np

#cargamos la bd de entrenamiento
db = np.loadtxt('/content/AirQuality.data',delimiter='\t')  # Assuming tab-delimiter
print ("Dim de la base de datos original: " + str(np.shape(db)))
db = db.reshape(9357,13)
DataBase = db
#La base d edatos AirQuality tiene muchos valores perdidos, vamos a quitar todos esos registros de la base de datos
#Están marcados como -200. Donde haya un valor -200 eliminaremos el registro.
j = 0
for i in range(0,np.size(db,0)):
    if -200 == db[i,12]:
        #print i
        j+=1
        DataBase = np.delete(DataBase,i,0)

print ("\nHay " + str(j) + " valores perdidos en la variable de salida. :(")

print ("\nDim de la base de datos sin las muestras con variable de salida perdido "+ str(np.shape(DataBase)))

#Ya hemos eliminado los registros con valor de la variable de salida perdido.

#Ahora vamos a imputar los valores perdidos en cada una de las características
print ("\nProcesando imputación de valores perdidos en las características . . .\n")

#data = open("AirQuality_cleaned.data", "w")

for k in range(0,np.size(DataBase,0)):
    for w in range(0,13):
        if -200 == DataBase[k,w]:
            DataBase[k,w] = round(np.mean(DataBase[:,w]))

    #data.write(str(DataBase[k,:]))

#data.close()
print ("Imputación finalizada.\n")

hay_missed_values = False
for i in range(0,np.size(DataBase,0)):
    if -200 in DataBase[i,:]:
        hay_missed_values = True
if(hay_missed_values):
    print ("Hay valores perdidos")
else:
    print ("No hay valores perdidos en la base de datos. Ahora se puede procesar. La base de datos está en la variable DataBase")

In [ ]:
X = DataBase[:,0:12]

Y = DataBase[:,12]*10 #(se multiplica por 100 para evitar errores en la interpretación de la eficiencia del modelo porque
                      #los valores de Y menores que 1 pueden llevar a malas interpretaciones).
Y = Y.reshape(np.size(X,0),1)
#La variable de salida es Humedad absoluta

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import Dense, Activation

#Error cuadrático medio (para problemas de regresión)
def ECM(Y_real,Y_est):
    N = np.size(Y_real)
    ecm = sum((Y_est - Y_real)**2)/N
    return ecm

#Para calcular el costo computacional
tiempo_i = time.time()

#Complete el código para crear la red neuronal artificial.
#Para la configuración de la red observe la tabla de más abajo.
#Recuerde que debe usar la función identidad en la capa de salida (linear).
#Para la capa oculta puede use relu.

model = Sequential()
model.add(keras.Input(shape=(12,)))
model.add(Dense(10, activation="relu"))
model.add(Dense(1, activation="linear"))

#Implemetamos la metodología de validación cross validation con 10 folds

Errores = np.ones(10)
j = 0
kf = KFold(n_splits=10)

for train_index, test_index in kf.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = Y[train_index], Y[test_index]

    #Complete el código
    #Use el optimizador Adam
    model.compile(loss='mean_absolute_error', optimizer='Adam', metrics=['mean_squared_error'])

    # Complete el código para entrenar el modelo
    model.fit(x=X_train,y=y_train, epochs=10 , batch_size=100, validation_split=0.1)

    #Complete el código aquí
    # Validación del modelo
    y_pred = model.predict(X_test)

    Errores[j] = ECM(y_test, y_pred)
    j+=1

print("\nError de validación: " + str(np.mean(Errores)) + " +/- " + str(np.std(Errores)))

print ("\n\nTiempo total de ejecución: " + str(time.time()-tiempo_i) + " segundos.")

3.1 Defina en sus palabras que es una época en el modelo de redes neuronales artificiales.

R/: :Una época es una pasada completa de todos los datos de entrenamiento por la red neuronal. En cada época, la red hace predicciones, calcula su error y ajusta sus pesos mediante backpropagation, entre mas epocas mas pasadas y en teoria se ajusta mejor el modelo.

3.2 Por qué se usa la función de activación lineal en lugar de reLu, tanh o sigmoid en la capa de salida para este problema?

R/: Debido a que tener una unica variable de salida implica una unica neurona, con una función de activación identidad ya que esta queda determinada por el tipo de valor que quiero definir, dicho de otra manera, esto significa que lo que llega sale. Necesito que salga un real y lo que llega es un real



3.3 Varíe el número de épocas del algoritmo de entrenamiento y el número de neuronas de la capa oculta y complete la siguiente tabla.

<table>
  <tr>
    <th># de Epocas</th>
    <th># Neuronas en capa oculta</th>
    <th>ECM +/- IC</th>
    <th>Tiempo de ejecución</th>
  </tr>
  <tr>
    <td>10</td>
    <td>10</td>
    <td>432.49107898891543 +/- 1101.6070985462402</td>
    <td>66.3002028465271</td>
  </tr>
  <tr>
    <td>10</td>
    <td>20</td>
    <td>68.5106813484473 +/- 79.8716169329648</td>
    <td>66.29425048828125</td>
  </tr>
  <tr>
    <td>100</td>
    <td>10</td>
    <td>61.124664075902594 +/- 73.08128249868363</td>
    <td>443.2412760257721</td>
  </tr>
</table>


3.4 Para esta base de datos con casi 10 mil registros, que puede decir del costo computacional del modelo con 100 épocas y 10 neuronas en la capa oculta

R/: El modelo con 100 épocas y 10 neuronas es muy costoso: tardó 443 segundos, casi 7 veces más que con 10 épocas, porque en cada época la red revisa los casi 10 mil registros. Aunque el error bajó bastante (de 432 a 61), no vale la pena, ya que con 10 épocas y 20 neuronas se logra un error parecido (68.5) en solo 66 segundos. Es decir, aumentar las neuronas mejora el modelo de forma más rápida que aumentar las épocas.

3.5 Haga un plot de las salidas reales y las predicciones por la RNA en la misma gráfica, para evaluar visualmente el desempeño de la red. Esto solo para la configuración en la cuál se encontró el menor error.

In [ ]:
# Gráfica: salidas reales vs predicciones
plt.figure(figsize=(14, 5))
plt.plot(y_test, label='Salida real', color='blue', linewidth=1.5)
plt.plot(y_pred, label='Predicción RNA', color='red', alpha=0.7)
plt.xlabel('Muestras de prueba')
plt.ylabel('Valor de salida')
plt.title('Salidas reales vs predicciones de la RNA (100 épocas, 10 neuronas)')
plt.legend()
plt.grid(True)
plt.show()